# Homework 3: Machine Learning for Classification

## Setup

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

## Data preparation

In [6]:
df = pd.read_csv('data/course_lead_scoring_2026.csv')
missing = df.isnull().sum()
missing[missing > 0]

lead_source          148
industry             240
employment_status    194
location             208
annual_income        369
lead_score            35
dtype: int64

In [25]:
categorical = list(df.select_dtypes(exclude='number').columns)
numerical = [c for c in df.columns if c not in categorical and c != 'converted']

print('categorical:', categorical)
print('numerical:', numerical)

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

df.isnull().sum().sum()

categorical: ['lead_source', 'industry', 'employment_status', 'location']
numerical: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


np.int64(0)

## Q1. Mode for industry

In [8]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [9]:
df['industry'].mode()[0]

'technology'

## Q2. Correlation matrix

In [10]:
corr = df[numerical].corr()
corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [14]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]

for a, b in pairs:
    print(a, b, round(corr.loc[a, b], 4))

best = max(pairs, key=lambda p: abs(corr.loc[p[0], p[1]]))
print(best)

interaction_count lead_score 0.9157
number_of_courses_viewed lead_score 0.7572
number_of_courses_viewed interaction_count 0.7216
annual_income interaction_count 0.1228
('interaction_count', 'lead_score')


## Split the data

In [12]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.pop('converted').values
y_val = df_val.pop('converted').values
y_test = df_test.pop('converted').values

print(len(df_train), len(df_val), len(df_test))

3000 1000 1000


## Q3. Mutual information

In [21]:
for col in categorical:
    score = mutual_info_score(y_train, df_train[col])
    print(col, round(score, 2))

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


## Q4. Logistic regression

In [16]:
def train_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)

    train_dicts = df_train[features].to_dict(orient='records')
    val_dicts = df_val[features].to_dict(orient='records')

    X_train = dv.fit_transform(train_dicts)
    X_val = dv.transform(val_dicts)

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

In [17]:
all_features = categorical + numerical

base_acc = train_and_score(all_features)
print('Accuracy:', round(base_acc, 2))

Accuracy: 0.65


## Q5. Feature elimination

In [24]:
for feature in all_features:
    features = [f for f in all_features if f != feature]
    acc = train_and_score(features)
    diff = base_acc - acc
    print(feature, round(acc, 4), 'diff:', round(diff, 4))

lead_source 0.642 diff: 0.003
industry 0.645 diff: 0.0
employment_status 0.645 diff: 0.0
location 0.645 diff: 0.0
annual_income 0.724 diff: -0.079
number_of_courses_viewed 0.643 diff: 0.002
interaction_count 0.601 diff: 0.044
lead_score 0.644 diff: 0.001


## Q6. Regularization

In [19]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    acc = train_and_score(all_features, C=C)
    print(C, round(acc, 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
